# Azerbaycan Öncesi/Sonrası Uydu Görüntüsü İndirme

Bu notebook'u **Google Colab**'da açıp sırayla hücreleri çalıştır (her hücrede Shift+Enter).
Google Earth Engine hesabına giriş isteyecek — Google hesabınla onaylaman yeterli.

Sonunda `.png` dosyaları doğrudan Colab'a iner, sol taraftaki **Files** (klasör ikonu) sekmesinden indirebilirsin.

In [ ]:
# 1) Gerekli kütüphaneleri kur
!pip install -q earthengine-api geemap

In [ ]:
# 2) Earth Engine'e giriş yap
import ee
import geemap

# Bu satır bir link verecek, tıkla, Google hesabınla giriş yap, kodu kopyala-yapıştır
ee.Authenticate()

# ÖNEMLİ: Aşağıdaki 'your-project-id' yerine kendi GEE proje adını yaz.
# Eğer daha önce hiç proje oluşturmadıysan, https://code.earthengine.google.com adresine
# gidip bir proje oluştur (ücretsiz), adını buraya yaz.
ee.Initialize(project='your-project-id')

In [ ]:
# 3) Görüntülenecek yerler (istediğin kadar ekleyebilirsin)
# format: 'isim': [boylam, enlem]
locations = {
    'Aghdam': [47.146, 39.987],
    # 'Aghali_Zangilan': [ENLEM_BURAYA, BOYLAM_BURAYA],  # Google Maps'ten bulup ekle
}

buffer_meters = 2000  # merkez noktadan kaç metre yarıçapında görüntü alınacak

before_start, before_end = '2019-05-01', '2019-09-30'
after_start, after_end = '2024-05-01', '2024-09-30'

vis_params = {'min': 0, 'max': 3000, 'gamma': 1.2, 'bands': ['B4', 'B3', 'B2']}

In [ ]:
# 4) Bulut-temiz kompozit oluşturan fonksiyon
def get_composite(region, start_date, end_date):
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .median()
    )
    return s2.select(['B4', 'B3', 'B2']).clip(region)

In [ ]:
# 5) Her yer için önce/sonra görüntülerini indir (.png olarak Colab'a kaydeder)
for name, (lon, lat) in locations.items():
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(buffer_meters).bounds()

    before_img = get_composite(region, before_start, before_end)
    after_img = get_composite(region, after_start, after_end)

    before_path = f'{name}_2019_before.png'
    after_path = f'{name}_2024_after.png'

    print(f'{name}: once (2019) indiriliyor...')
    geemap.get_image_thumbnail(before_img, before_path, vis_params, dimensions=800, region=region)

    print(f'{name}: sonra (2024) indiriliyor...')
    geemap.get_image_thumbnail(after_img, after_path, vis_params, dimensions=800, region=region)

    print(f'{name} tamamlandi: {before_path}, {after_path}')

print('\nHepsi bitti. Sol taraftaki Files (klasor ikonu) sekmesinden .png dosyalarini indirebilirsin.')

In [ ]:
# 6) (Opsiyonel) Görüntüleri burada, notebook icinde de gorebilirsin
from IPython.display import Image, display

for name in locations.keys():
    print(f'--- {name}: ONCE (2019) ---')
    display(Image(f'{name}_2019_before.png'))
    print(f'--- {name}: SONRA (2024) ---')
    display(Image(f'{name}_2024_after.png'))

## Notlar

- Görüntü çok bulutlu çıkarsa, `before_start/before_end` veya `after_start/after_end` tarih aralığını biraz kaydır (örn. `'2019-06-01'` - `'2019-08-31'`).
- `ee.Initialize(project='your-project-id')` satırındaki proje adını mutlaka kendi GEE proje adınla değiştir, yoksa hata alırsın.
- Yeni bir yer eklemek için `locations` sözlüğüne yeni bir satır ekle: `'Isim': [boylam, enlem],`
- Koordinatları Google Maps'te ilgili yere sağ tıklayıp çıkan enlem/boylamdan alabilirsin (dikkat: Google Maps enlem,boylam sırasında gösterir; koddaki liste ise boylam,enlem sırasında).